In [1]:
from target_benchmark.evaluators import TARGET, get_task_names
from target_benchmark.retrievers import AbsCustomEmbeddingRetriever, LlamaIndexRetriever, AbsStandardEmbeddingRetriever

import numpy as np
from typing import List, Tuple, Dict, Iterable
from sentence_transformers import SentenceTransformer
import pandas as pd
import json
from pathlib import Path
import datamart_profiler
import chromadb
from openai import OpenAI
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import os
import re
import time
from itertools import islice


from concurrent.futures import ThreadPoolExecutor, as_completed



import sys
sys.path.append(r'C:\Users\rite\Documents\dataset_discovery_llm_implementation\Dataset-search-LLM\dataset_preprocessing')
import preprocessing_utils as utils

Initialization of Chroma DB 


In [51]:
client = chromadb.PersistentClient(path="/chroma_db")

embedding_function = utils.OpenAIEmbeddingFunction()

collection = client.get_or_create_collection(
    name="test_instructions_ottqa",
    embedding_function=OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name="text-embedding-3-small"
    )
)

In [6]:
#list of dictionaries to hold IDs and profiles
metadata = []
model = "gpt-5-nano-2025-08-07"

Loading preprocessed queries from FetaQA DB


In [4]:
query_preprocessing = []
queries_result = []


with open("query_optimization_result.json", "r", encoding="utf-8") as f:
    query_preprocessing = json.load(f)

for batch in query_preprocessing:
    for result in batch:
        list_of_tuples = []
        for tup in result[1]:
            list_of_tuples.append((tup[0], tup[1]))
        queries_result.append((result[0], list_of_tuples))

queries_result_dict = dict(queries_result)

In [14]:
queries_result_1 = {}
queries_result_3 = {}
queries_result_5 = {}
queries_result_10 = {}

with open("query_optimization_ottqa_result_recall_1_3_5_10.json", "r", encoding="utf-8") as f:
    query_preprocessing = json.load(f)

for batch in query_preprocessing:
    for group_of_queries in batch:
        queries_result_1[group_of_queries[0]] = group_of_queries[1].get("1")
        queries_result_3[group_of_queries[0]] = group_of_queries[1].get("3")
        queries_result_5[group_of_queries[0]] = group_of_queries[1].get("5")
        queries_result_10[group_of_queries[0]] = group_of_queries[1].get("10")


for key, value in queries_result_1.items():
    queries_result_1[key] = [tuple(x) for x in value]   

for key, value in queries_result_3.items():
    queries_result_3[key] = [tuple(x) for x in value] 

for key, value in queries_result_5.items():
    queries_result_5[key] = [tuple(x) for x in value] 

for key, value in queries_result_10.items():
    queries_result_10[key] = [tuple(x) for x in value] 

In [ ]:
queries_result_nosem_1 = {}
queries_result_nosem_3 = {}
queries_result_nosem_5 = {}
queries_result_nosem_10 = {}

with open("query_optimization_ottqa_result_nosem_recall_1_3_5_10.json", "r", encoding="utf-8") as f:
    query_preprocessing = json.load(f)

for batch in query_preprocessing:
    for group_of_queries in batch:
        queries_result_nosem_1[group_of_queries[0]] = group_of_queries[1].get("1")
        queries_result_nosem_3[group_of_queries[0]] = group_of_queries[1].get("3")
        queries_result_nosem_5[group_of_queries[0]] = group_of_queries[1].get("5")
        queries_result_nosem_10[group_of_queries[0]] = group_of_queries[1].get("10")


for key, value in queries_result_nosem_1.items():
    queries_result_nosem_1[key] = [tuple(x) for x in value]
    print(queries_result_nosem_1)   

for key, value in queries_result_nosem_3.items():
    queries_result_nosem_3[key] = [tuple(x) for x in value] 

for key, value in queries_result_nosem_5.items():
    queries_result_nosem_5[key] = [tuple(x) for x in value] 

for key, value in queries_result_nosem_10.items():
    queries_result_nosem_10[key] = [tuple(x) for x in value]

Custom Parallel Retriever with multithread support to speed up batch processing

In [33]:
class customParallelRetriever(AbsCustomEmbeddingRetriever):
    def __init__(self, **kwargs):
        super().__init__(expected_corpus_format="nested array")

    '''
    def embed_batch(self, batch):
        start_time = time.time()
        for table_id, database_id, table, context in zip(batch['table_id'], batch['database_id'], batch['table'], batch['context']):
            table = pd.DataFrame(table)
            entry = {
                'table_id': table_id,
                'database_id': database_id,
                'title': context.get("title"),
                'data_profile': utils.dataframe_to_data_profile(table, context),
                'semantic_profile': '',
                'instructions': [],
            }

            entry['semantic_profile'] = utils.call_openai_api(
                utils.generate_prompt_semantic_profile(entry['data_profile']),
                model
            )

            prompt = utils.generate_prompt_instructions(entry['title'], entry['data_profile'], entry['semantic_profile'])
            entry["instructions"] = utils.json_to_dict(utils.call_openai_api(prompt, model))

            instructions = entry['instructions']['queries']
            for i, instruction in enumerate(instructions):
                collection.add(
                    documents=[instruction["query"]],
                    ids=[f"instruction_{i}_{entry['title']}"],
                    metadatas={
                        "title": entry['title'],
                        "table_id": entry['table_id'],
                        "database_id": entry['database_id'],
                        "semantic_profile": entry['semantic_profile'],
                        "data_profile": entry['data_profile']
                    }
                )
            end_time = time.time()
        elapsed = end_time - start_time
        print(f"Batch completato in {elapsed:.2f} secondi")
        return True
    '''

    def embed_corpus(self, dataset_name: str, corpus: Iterable[Dict]) -> None:
        #corpus already embedded in db in precedent runs
        '''
        start_time = time.time()

        #limited_corpus = islice(corpus, 4)
            
        with ThreadPoolExecutor(max_workers=10) as executor:
            futures = [executor.submit(self.embed_batch, batch) for batch in corpus]
            for f in as_completed(futures):
                try:
                    f.result()
                except Exception as e:
                    print(f"Errore nel batch: {e}")            

        end_time = time.time()
        elapsed = end_time - start_time
        print(f"4 Batch completati in {elapsed:.2f} secondi")
        '''    

    def retrieve(self, query: str, dataset_name: str, top_k: int) -> List[Tuple]:
        """
        Here we retrieve the most similar dataset given a query.
        First we optimize the query, then we retrieve.
        We compute the embedding of the query and we search in the vector DB.
        Must return a list of (database_id, table_id) tuples.
        """
        #return utils.get_subqueries_from_query(query, model, collection, top_k) run this without preprocessing
        #result = queries_result_10.get(query) #run this for recall@10
        result = queries_result_5.get(query) #run this for recall@5
        #result = queries_result_3.get(query) #run this for recall@3
        #result = queries_result_1.get(query) #run this for recall@1
        return result
        

Evaluations script

In [ ]:
target_fetaqa = TARGET(("Table Retrieval Task", "ottqa"), persist_log=False)
retriever = customParallelRetriever(model_name="all-mpnet-base-v2")
performance = target_fetaqa.run(retriever=retriever, batch_size=10, split="validation", top_k=10)

Retrieving Tables for ottqa...: 100%|██████████| 2214/2214 [00:00<00:00, 79129.04it/s]


In [35]:
print(performance)

{'Table Retrieval Task': {'ottqa': {'retrieval_performance': {'k': 5, 'accuracy': 0.5483288166214996, 'precision': None, 'recall': 0.5483288166214996, 'capped_recall': 0.5483288166214996, 'retrieval_duration_process': 0.0, 'avg_retrieval_duration_process': 0.0, 'retrieval_duration_wall_clock': 0.00624, 'avg_retrieval_duration_wall_clock': 0.0}, 'downstream_task_performance': {'task_name': None, 'scores': None}, 'embedding_statistics': {'embedding_creation_duration_process': 0.0, 'avg_embedding_creation_duration_process': 0.0, 'embedding_creation_duration_wall_clock': 0.0, 'avg_embedding_creation_duration_wall_clock': 0.0, 'embedding_size': 0.0, 'avg_embedding_size': 0.0}}}}


Ablation studies

1. No query optimization
2. No semantic profile
3. No query optimization and no semantic profile

1. No query optimization:

In [52]:
class customParallelRetriever_noQueryOptimization(AbsCustomEmbeddingRetriever):
    def __init__(self, **kwargs):
        super().__init__(expected_corpus_format="nested array")

    def embed_corpus(self, dataset_name: str, corpus: Iterable[Dict]) -> None:
        #corpus already embedded in db in precedent runs
        print('corpus already embedded in db in precedent runs')    



    def retrieve(self, query: str, dataset_name: str, top_k: int) -> List[Tuple]:
        """
        In this version we use plain queries without optimization.
        """

        result = []
        
        metadatas = collection.query(
        query_texts=query,
        include=["metadatas"],
        n_results=100
        )

        for subquery_results_metadata in metadatas['metadatas']:
            for single_dataset_metadata in subquery_results_metadata:
                tuple_result = (single_dataset_metadata['database_id'], single_dataset_metadata['table_id'])
                result.append(tuple_result)

        return utils.top_k_tuples(result, top_k)
        

In [57]:
target_fetaqa_no_query_opt = TARGET(("Table Retrieval Task", "ottqa"), persist_log=False)
retriever = customParallelRetriever_noQueryOptimization(model_name="all-mpnet-base-v2")
performance = target_fetaqa_no_query_opt.run(retriever=retriever, batch_size=10, split="validation", top_k=1)

corpus already embedded in db in precedent runs


Retrieving Tables for ottqa...: 100%|██████████| 2214/2214 [11:53<00:00,  3.10it/s]


In [58]:
print(performance)

{'Table Retrieval Task': {'ottqa': {'retrieval_performance': {'k': 1, 'accuracy': 0.28906955736224027, 'precision': None, 'recall': 0.28906955736224027, 'capped_recall': 0.28906955736224027, 'retrieval_duration_process': 42.17188, 'avg_retrieval_duration_process': 0.01905, 'retrieval_duration_wall_clock': 712.59329, 'avg_retrieval_duration_wall_clock': 0.32186}, 'downstream_task_performance': {'task_name': None, 'scores': None}, 'embedding_statistics': {'embedding_creation_duration_process': 0.0, 'avg_embedding_creation_duration_process': 0.0, 'embedding_creation_duration_wall_clock': 0.0, 'avg_embedding_creation_duration_wall_clock': 0.0, 'embedding_size': 0.0, 'avg_embedding_size': 0.0}}}}


2. No Semantic Profile:

In [72]:
collection_no_sem = client.get_or_create_collection(
    name="test_instructions_ottqa_no_sem_profile",
    embedding_function=OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        model_name="text-embedding-3-small"
    )
)

In [73]:
class customParallelRetriever_no_semantic_profile(AbsCustomEmbeddingRetriever):
    def __init__(self, **kwargs):
        super().__init__(expected_corpus_format="nested array")

    '''
    def embed_batch(self, batch):
        start_time = time.time()
        for table_id, database_id, table, context in zip(batch['table_id'], batch['database_id'], batch['table'], batch['context']):
            table = pd.DataFrame(table)
            entry = {
                'table_id': table_id,
                'database_id': database_id,
                'title': context.get("title"),
                'data_profile': utils.dataframe_to_data_profile(table, context),
                'instructions': [],
            }

            prompt = utils.generate_prompt_instructions_no_semantic_profile(entry['title'], entry['data_profile'])
            entry["instructions"] = utils.json_to_dict(utils.call_openai_api(prompt, model))

            instructions = entry['instructions']['queries']
            for i, instruction in enumerate(instructions):
                collection_no_sem.add(
                    documents=[instruction["query"]],
                    ids=[f"instruction_{i}_{entry['title']}"],
                    metadatas={
                        "title": entry['title'],
                        "table_id": entry['table_id'],
                        "database_id": entry['database_id'],
                        "data_profile": entry['data_profile']
                    }
                )
            end_time = time.time()
        elapsed = end_time - start_time
        print(f"Batch completato in {elapsed:.2f} secondi")
        return True
        '''

    def embed_corpus(self, dataset_name: str, corpus: Iterable[Dict]) -> None:        
        '''
        start_time = time.time()            

        with ThreadPoolExecutor(max_workers=8) as executor:
            futures = [executor.submit(self.embed_batch, batch) for batch in corpus]
            for f in as_completed(futures):
                try:
                    f.result()
                except Exception as e:
                    print(f"Errore nel batch: {e}")
                    print(f"Status code: {e.response.status_code}")
                    print(f"Dettagli: {e.response.text}")
                    print(f"Headers: {e.response.headers}")
                    

        end_time = time.time()
        elapsed = end_time - start_time
        print(f"4 Batch completati in {elapsed:.2f} secondi")  
        '''


    def retrieve(self, query: str, dataset_name: str, top_k: int) -> List[Tuple]:
        """
        Here we retrieve the most similar dataset given a query.
        First we optimize the query, then we retrieve.
        We compute the embedding of the query and we search in the vector DB.
        Must return a list of (database_id, table_id) tuples.
        """
        #return utils.get_subqueries_from_query(query, model, collection, top_k)
        result = queries_result_nosem_1.get(query)
        #result = result = queries_result_10.get(query) #run this for recall@10
        return result

In [74]:
target_ottqa_no_sem = TARGET(("Table Retrieval Task", "ottqa"), persist_log=False)
retriever_no_sem = customParallelRetriever_no_semantic_profile(model_name="all-mpnet-base-v2")
performance = target_ottqa_no_sem.run(retriever=retriever_no_sem, batch_size=10, split="validation", top_k=1)

ValidationError: 1 validation error for RetrievalResultDataModel
retrieval_results
  Input should be a valid list [type=list_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/list_type

In [46]:
print(performance)

{'Table Retrieval Task': {'ottqa': {'retrieval_performance': {'k': 10, 'accuracy': 0.6991869918699187, 'precision': None, 'recall': 0.6991869918699187, 'capped_recall': 0.6991869918699187, 'retrieval_duration_process': 0.09375, 'avg_retrieval_duration_process': 4e-05, 'retrieval_duration_wall_clock': 0.15599, 'avg_retrieval_duration_wall_clock': 7e-05}, 'downstream_task_performance': {'task_name': None, 'scores': None}, 'embedding_statistics': {'embedding_creation_duration_process': 1203.42188, 'avg_embedding_creation_duration_process': 1.52525, 'embedding_creation_duration_wall_clock': 3298.58865, 'avg_embedding_creation_duration_wall_clock': 4.18072, 'embedding_size': 1847.61139, 'avg_embedding_size': 2.34171}}}}


3. No semantic profile and no query optimization

In [60]:
class customParallelRetriever_noQueryOptimization_noSem(AbsCustomEmbeddingRetriever):
    def __init__(self, **kwargs):
        super().__init__(expected_corpus_format="nested array")

    def embed_corpus(self, dataset_name: str, corpus: Iterable[Dict]) -> None:
        #corpus already embedded in db in precedent runs
        print('corpus already embedded in db in precedent runs')    



    def retrieve(self, query: str, dataset_name: str, top_k: int) -> List[Tuple]:
        """
        In this version we use plain queries without optimization.
        """

        result = []
        
        metadatas = collection_no_sem.query(
        query_texts=query,
        include=["metadatas"],
        n_results=100
        )

        for subquery_results_metadata in metadatas['metadatas']:
            for single_dataset_metadata in subquery_results_metadata:
                tuple_result = (single_dataset_metadata['database_id'], single_dataset_metadata['table_id'])
                result.append(tuple_result)

        return utils.top_k_tuples(result, top_k)
        

In [65]:
target_ottqa_no_query_opt_no_sem = TARGET(("Table Retrieval Task", "ottqa"), persist_log=False)
retriever_no_sem = customParallelRetriever_noQueryOptimization_noSem(model_name="all-mpnet-base-v2")
performance = target_ottqa_no_query_opt_no_sem.run(retriever=retriever_no_sem, batch_size=10, split="validation", top_k=1)

corpus already embedded in db in precedent runs


Retrieving Tables for ottqa...: 100%|██████████| 2214/2214 [10:29<00:00,  3.52it/s]


In [66]:
print(performance)

{'Table Retrieval Task': {'ottqa': {'retrieval_performance': {'k': 1, 'accuracy': 0.35591689250225833, 'precision': None, 'recall': 0.35591689250225833, 'capped_recall': 0.35591689250225833, 'retrieval_duration_process': 28.625, 'avg_retrieval_duration_process': 0.01293, 'retrieval_duration_wall_clock': 628.71496, 'avg_retrieval_duration_wall_clock': 0.28397}, 'downstream_task_performance': {'task_name': None, 'scores': None}, 'embedding_statistics': {'embedding_creation_duration_process': 0.0, 'avg_embedding_creation_duration_process': 0.0, 'embedding_creation_duration_wall_clock': 0.0, 'avg_embedding_creation_duration_wall_clock': 0.0, 'embedding_size': 0.0, 'avg_embedding_size': 0.0}}}}
